# D5 — APR from CPRA

**Goal.** Produce a Berkeley Housing APR from CPRA-fulfilled `BP_Annual Permit Report` data, without depending on `v2.permits`.

**Architectural pivot.** CPRA is the source of truth for permit-level data. The existing `databases/berkeley_housing_v2.db` is used only for the **curated project list** (project_id, address, APN, stage) — not for permit records.

**Output:** `output/D5/table_a2_CY{year}.csv` for CY 2018-2025, plus an audit log and a comparison summary.

## Caveats (documented up front)

1. **CPRA files are scoped by `Finaled Date`** for the file boundaries, but the union of both files captures permits regardless of submittal date. The data is comprehensive for permits Finaled 2018-2025. Permits filed earlier that finaled in this window appear here.
2. **Entitlement data (ENT stage) is not in CPRA** — Planning module data is a separate request. APR Table A2's entitlement columns will be null/empty for now.
3. **Income category breakdown (affordability) is not in CPRA** — requires separate enrichment from inclusionary housing records or project-level affordability data.
4. **The CPRA data uses Berkeley's structured `Work Type` field**, not inferred from description text. This is more authoritative than v2's `permit_type` but is still subject to data entry by Berkeley staff.
5. **`TENURE` defaults to `'Renter'`** in the output table — CPRA does not include tenure; this is a placeholder requiring separate enrichment.


## Cell 1 — Imports + paths

In [1]:
import pandas as pd
import sqlite3
from pathlib import Path

ROOT = Path("/Users/johngage/berkeley-data")
CPRA_DIR = ROOT / "data/raw/cpra-downloads"
CPRA_2018 = CPRA_DIR / "BP_Annual Permit Report-2018-2022.xlsx"
CPRA_2023 = CPRA_DIR / "BP_Annual Permit Report-2023-2025.xlsx"
V2_DB = ROOT / "databases/berkeley_housing_v2.db"
V1_APR_2025 = ROOT / "data/apr/2025/table_a2_2025.csv"   # v1-derived baseline, for Cell 9 diff
OUT_DIR = ROOT / "output/D5"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Paths configured. OUT_DIR:", OUT_DIR)

Paths configured. OUT_DIR: /Users/johngage/berkeley-data/output/D5


## Cell 2 — Load both XLSXs, union, dedup

Header is at row 8 (1-indexed) of each XLSX, so `header=7` for pandas (0-indexed).

In [2]:
df_a = pd.read_excel(CPRA_2018, sheet_name="BP_Annual Permit Report", header=7)
df_b = pd.read_excel(CPRA_2023, sheet_name="BP_Annual Permit Report", header=7)
print(f"2018-2022 raw: {len(df_a)} rows")
print(f"2023-2025 raw: {len(df_b)} rows")

df = pd.concat([df_a, df_b], ignore_index=True)
print(f"union (before dedup): {len(df)} rows")

# When the same PermitNumber appears in both files (1,430 overlapping permits per inventory),
# keep the row with the later Finaled Date (or later Submittal Date as tiebreaker).
df["__finaled_sort"] = pd.to_datetime(df["Finaled Date"], errors="coerce")
df["__submit_sort"] = pd.to_datetime(df["Submittal Date"], errors="coerce")
df = (df.sort_values(["__finaled_sort", "__submit_sort"], na_position="first")
        .drop_duplicates(subset=["PermitNumber"], keep="last")
        .drop(columns=["__finaled_sort", "__submit_sort"]))
df = df.dropna(subset=["PermitNumber"])
df = df[df["PermitNumber"].astype(str).str.strip() != ""]
print(f"after dedup on PermitNumber + null-drop: {len(df)} unique permits")

2018-2022 raw: 18053 rows
2023-2025 raw: 14149 rows
union (before dedup): 32202 rows
after dedup on PermitNumber + null-drop: 30764 unique permits


/var/folders/zr/1lcy71z97n33bq1zyg3vtbp80000gn/T/ipykernel_19222/3606989215.py:6: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df_a, df_b], ignore_index=True)


## Cell 3 — Clean and normalize

Strip whitespace, convert dates and numerics, drop nulls.

In [3]:
str_cols = ["PermitNumber", "Issuance Status", "Finaled Status", "Completed",
            "Parcel Number", "StreetName", "StreetType", "WorkDescription",
            "ADU", "Detached", "Work Type", "OccType", "SubType", "CO Required"]
for c in str_cols:
    if c in df.columns:
        df[c] = df[c].astype(str).str.strip()
        df[c] = df[c].replace({"nan": None, "None": None, "": None})

date_cols = ["Submittal Date", "Issuance Date", "Finaled Date", "Completed Date"]
for c in date_cols:
    if c in df.columns:
        df[c] = pd.to_datetime(df[c], errors="coerce")

num_cols = ["JobValuation", "NumberUnits", "UnitsAdded", "UnitsRemoved", "StreetNumber"]
for c in num_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

print(f"normalized {len(df)} rows")
print(f"  Submittal Date populated: {df['Submittal Date'].notna().sum()}/{len(df)}")
print(f"  Finaled Date populated:   {df['Finaled Date'].notna().sum()}/{len(df)}")
print(f"  UnitsAdded > 0:           {(df['UnitsAdded'].fillna(0) > 0).sum()}/{len(df)}")

normalized 30764 rows
  Submittal Date populated: 30764/30764
  Finaled Date populated:   21632/30764
  UnitsAdded > 0:           1605/30764


## Cell 4 — Filter to housing-relevant subset

Inclusion rules (UNION):
- `Work Type IN ('New', 'Addition', 'Addition/Alteration')` AND `SubType IN ('Residential', 'Mixed Use')`, OR
- `ADU == 'Yes'`, OR
- `UnitsAdded > 0 OR UnitsRemoved > 0`

In [4]:
work_type_rule = df["Work Type"].isin(["New", "Addition", "Addition/Alteration"]) & df["SubType"].isin(["Residential", "Mixed Use"])
adu_rule = df["ADU"] == "Yes"
units_rule = (df["UnitsAdded"].fillna(0) > 0) | (df["UnitsRemoved"].fillna(0) > 0)
keep = work_type_rule | adu_rule | units_rule
df_housing = df[keep].copy()

print(f"input:   {len(df)} rows")
print(f"kept:    {len(df_housing)}")
print(f"dropped: {len(df) - len(df_housing)}")
print()
print("Work Type breakdown of kept:")
print(df_housing["Work Type"].value_counts(dropna=False).to_string())
print()
print(f"ADU=Yes count in kept: {(df_housing['ADU'] == 'Yes').sum()}")

input:   30764 rows
kept:    6630
dropped: 24134

Work Type breakdown of kept:
Work Type
Alteration             2286
New                    1595
Addition/Alteration    1591
None                    539
Addition                509
Demolition               67
Sign                     43

ADU=Yes count in kept: 3219


## Cell 5 — Group by Parcel Number, identify master

Master identification (CPRA-shaped, distinct from the v2 CO rule's description-pattern approach):
1. Largest `UnitsAdded > 0`, tiebreak by earliest `Submittal Date`
2. Fallback: `Work Type == 'New'`, then largest `JobValuation`, then earliest `Submittal Date`

In [5]:
df_housing["Parcel Number"] = df_housing["Parcel Number"].astype(str).str.strip()
df_housing = df_housing[df_housing["Parcel Number"].notna() &
                         (df_housing["Parcel Number"] != "nan") &
                         (df_housing["Parcel Number"] != "")]
parcels = df_housing.groupby("Parcel Number")
print(f"distinct parcels with housing-relevant permits: {parcels.ngroups}")

projects = []
no_master = 0
for apn, group in parcels:
    # Parcel-collapse fix (same-year gated): when >=2 independent New-with-units
    # permits share an issuance year, emit one project per sibling. Cross-year and
    # single permits fall through to the original single-master logic (else branch).
    is_child = (group["PermitNumber"].astype(str).str.contains("-REV", na=False, regex=False) |
                group["PermitNumber"].astype(str).str.contains("-DEF", na=False, regex=False))
    sib_all = group[(group["Work Type"] == "New") &
                    (group["UnitsAdded"].fillna(0) > 0) & (~is_child)].copy()
    sib_all["__iy"] = pd.to_datetime(sib_all["Issuance Date"], errors="coerce").dt.year
    yr_counts = sib_all.groupby("__iy")["PermitNumber"].nunique()
    sibling_years = set(yr_counts[yr_counts >= 2].index)
    if sibling_years:
        siblings = sib_all[sib_all["__iy"].isin(sibling_years)]
        for _, master in siblings.iterrows():
            subs = group[group["PermitNumber"] != master["PermitNumber"]].copy()
            projects.append({"apn": apn, "master": master, "subs": subs, "all": group})
    else:
        with_units = group[group["UnitsAdded"].fillna(0) > 0]
        if len(with_units) > 0:
            master = with_units.sort_values(["UnitsAdded", "Submittal Date"],
                                             ascending=[False, True], na_position="last").iloc[0]
        else:
            new_only = group[group["Work Type"] == "New"]
            cand = new_only if len(new_only) > 0 else group
            cand = cand.sort_values(["JobValuation", "Submittal Date"],
                                     ascending=[False, True], na_position="last")
            if len(cand) == 0:
                no_master += 1
                continue
            master = cand.iloc[0]
        subs = group[group["PermitNumber"] != master["PermitNumber"]].copy()
        projects.append({"apn": apn, "master": master, "subs": subs, "all": group})

print(f"total projects identified: {len(projects)}")
print(f"parcels skipped (no master found): {no_master}")

distinct parcels with housing-relevant permits: 4078


total projects identified: 4098
parcels skipped (no master found): 0


## Cell 6 — Join curated v2 project list

Only project metadata is pulled from v2 (project_id, name, address, APN, stage). No permit data is pulled. The APN format matches between CPRA (`055 183500901`) and v2.parcels, so it's a clean string join.

In [6]:
con = sqlite3.connect(f"file:{V2_DB}?mode=ro&immutable=1", uri=True)
v2_proj_df = pd.read_sql_query("""
    SELECT p.id AS project_id, p.canonical_name AS name, p.canonical_address AS address,
           pa.apn,
           vst.code AS stage
    FROM projects p
    LEFT JOIN project_parcels pp ON pp.project_id = p.id
    LEFT JOIN parcels pa ON pa.id = pp.parcel_id
    LEFT JOIN vocabulary_stage_types vst ON p.current_stage_type_id = vst.id
""", con)
con.close()

v2_apns = set(v2_proj_df["apn"].dropna().astype(str).str.strip())
cpra_apns = set(p["apn"] for p in projects)
in_v2 = cpra_apns & v2_apns
new_apns = cpra_apns - v2_apns
v2_only = v2_apns - cpra_apns

print(f"v2 project_parcels-joined rows:    {len(v2_proj_df)}")
print(f"distinct v2 APNs:                  {len(v2_apns)}")
print(f"distinct CPRA project APNs:        {len(cpra_apns)}")
print(f"IN_V2 (CPRA ∩ v2):                 {len(in_v2)}")
print(f"NEW (CPRA only):                   {len(new_apns)}")
print(f"v2-only (v2 not in CPRA-housing):  {len(v2_only)}")

# Build join dict apn -> v2 project info (first match if multiple)
v2_by_apn = {}
for _, row in v2_proj_df.iterrows():
    apn = str(row["apn"]).strip() if pd.notna(row["apn"]) else None
    if apn and apn not in v2_by_apn:
        v2_by_apn[apn] = {"project_id": row["project_id"], "name": row["name"],
                          "address": row["address"], "stage": row["stage"]}

v2 project_parcels-joined rows:    181
distinct v2 APNs:                  171
distinct CPRA project APNs:        4078
IN_V2 (CPRA ∩ v2):                 46
NEW (CPRA only):                   4032
v2-only (v2 not in CPRA-housing):  125


## Cell 7 — Compute APR fields per project

For each project (one master permit per parcel):
- `bp_issue_date = master.Issuance Date`
- `bp_units = master.UnitsAdded − master.UnitsRemoved`
- `co_date = master.Finaled Date` (if `Finaled Status == 'Finaled'`)
- `co_units = bp_units + sum(rev_finaled.UnitsAdded − rev_finaled.UnitsRemoved)` — reconciles sub-permit REVs that altered unit counts

Sub-permit REVs are identified by `PermitNumber` startswith `{master.PermitNumber}-`.

In [7]:
records = []
for p in projects:
    m = p["master"]
    subs = p["subs"]
    apn = p["apn"]
    master_num = m["PermitNumber"]

    rev_prefix = f"{master_num}-"
    rev_subs = subs[subs["PermitNumber"].astype(str).str.startswith(rev_prefix)]
    rev_finaled = rev_subs[rev_subs["Finaled Status"] == "Finaled"]

    bp_units = (m["UnitsAdded"] if pd.notna(m["UnitsAdded"]) else 0) - \
               (m["UnitsRemoved"] if pd.notna(m["UnitsRemoved"]) else 0)

    # Cause 2 fix (2026-05-29): floor at 0 for Alteration/Demolition/Add-Alt masters with
    # net-negative units. Berkeley reports demolitions as a separate aggregate (HCD's
    # "Number of Demolished/Destroyed Units" column); D5's net-out approach mixed accounting
    # models. See docs/audit/2026-05-29_causes_2_3_diagnostic.md.
    if m["Work Type"] in ("Alteration", "Demolition", "Addition/Alteration") and bp_units < 0:
        bp_units = 0

    bp_issue_date = m["Issuance Date"] if pd.notna(m["Issuance Date"]) else None
    finaled = (m["Finaled Status"] == "Finaled")
    co_date = m["Finaled Date"] if finaled and pd.notna(m["Finaled Date"]) else None

    if co_date:
        co_units = bp_units  # master-only; verified cumulative-restatement convention CY 2024
    else:
        co_units = None

    v2_match = v2_by_apn.get(apn)
    addr = f"{int(m['StreetNumber']) if pd.notna(m['StreetNumber']) else ''} " \
           f"{m['StreetName'] or ''} {m['StreetType'] or ''}".strip()

    records.append({
        "apn": apn,
        "master_permit": master_num,
        "master_work_type": m["Work Type"],
        "master_subtype": m["SubType"],
        "master_occtype": m["OccType"],
        "address": addr,
        "is_adu": (m["ADU"] == "Yes") and (m["Work Type"] == "New"),  # Cause 3 fix (2026-05-29)
        "submittal_date": m["Submittal Date"],
        "bp_issue_date": bp_issue_date,
        "bp_units": float(bp_units) if pd.notna(bp_units) else None,
        "co_date": co_date,
        "co_units": float(co_units) if co_units is not None else None,
        "reporting_year_bp": bp_issue_date.year if bp_issue_date else None,
        "reporting_year_co": co_date.year if co_date else None,
        "v2_project_id": v2_match["project_id"] if v2_match else None,
        "v2_name": v2_match["name"] if v2_match else None,
        "v2_address": v2_match["address"] if v2_match else None,
        "v2_stage": v2_match["stage"] if v2_match else None,
        "in_v2": v2_match is not None,
        "rev_sub_count": len(rev_subs),
        "rev_finaled_count": len(rev_finaled),
        "job_valuation": m["JobValuation"] if pd.notna(m["JobValuation"]) else None,
        "work_description": m["WorkDescription"],
    })
proj_df = pd.DataFrame(records)
print(f"total project records: {len(proj_df)}")
print(f"  with BP year: {proj_df['reporting_year_bp'].notna().sum()}")
print(f"  with CO year: {proj_df['reporting_year_co'].notna().sum()}")
print(f"  in_v2=True:   {proj_df['in_v2'].sum()}")
print()
print("BP year histogram:")
print(proj_df['reporting_year_bp'].value_counts().sort_index().to_string())
print()
print("CO year histogram:")
print(proj_df['reporting_year_co'].value_counts().sort_index().to_string())

total project records: 4098
  with BP year: 4073
  with CO year: 3093
  in_v2=True:   47

BP year histogram:
reporting_year_bp
2015.0      2
2016.0     75
2017.0    182
2018.0    254
2019.0    211
2020.0    201
2021.0    252
2022.0    477
2023.0    694
2024.0    899
2025.0    824
2026.0      2

CO year histogram:
reporting_year_co
2018.0    209
2019.0    227
2020.0    153
2021.0    205
2022.0    336
2023.0    536
2024.0    697
2025.0    730


## Cell 7b — Cycle-aware classification (additive)

Adds `bp_cycle`, `co_cycle`, `bp_in_projection_period`, `co_in_projection_period` to `proj_df`. Classifiers are pure functions from `scripts.housing_rules`. Lookups in that package carry the citation for every date used here.

In [8]:
# Cycle-aware classification — additive columns derived from bp_issue_date / co_date.
# Classifiers: scripts.housing_rules.classifiers.{cycle_for_date, is_projection_period}.
# Lookups: scripts.housing_rules.lookups (CITATIONS section documents every date).

import sys
from pathlib import Path as _Path

# Walk up from CWD until we find the scripts/housing_rules package, then add to sys.path.
_p = _Path.cwd()
while not (_p / "scripts" / "housing_rules").exists() and _p.parent != _p:
    _p = _p.parent
if not (_p / "scripts" / "housing_rules").exists():
    raise RuntimeError(f"Could not locate scripts/housing_rules/ from {_Path.cwd()}")
if str(_p) not in sys.path:
    sys.path.insert(0, str(_p))

from scripts.housing_rules.classifiers import cycle_for_date, is_projection_period

def _to_date(x):
    """Convert pd.Timestamp / datetime / None / NaT to datetime.date or None."""
    if x is None or pd.isna(x):
        return None
    if hasattr(x, "date"):
        return x.date()
    return x

proj_df["bp_cycle"]                = proj_df["bp_issue_date"].apply(lambda x: cycle_for_date(_to_date(x)))
proj_df["co_cycle"]                = proj_df["co_date"].apply(lambda x: cycle_for_date(_to_date(x)))
proj_df["bp_in_projection_period"] = proj_df["bp_issue_date"].apply(lambda x: is_projection_period(_to_date(x)))
proj_df["co_in_projection_period"] = proj_df["co_date"].apply(lambda x: is_projection_period(_to_date(x)))

print("bp_cycle distribution:")
print(proj_df["bp_cycle"].value_counts(dropna=False).to_string())
print()
print("co_cycle distribution:")
print(proj_df["co_cycle"].value_counts(dropna=False).to_string())
print()
print(f"Projection period flags (bp): {proj_df['bp_in_projection_period'].sum()} rows")
print(f"Projection period flags (co): {proj_df['co_in_projection_period'].sum()} rows")


bp_cycle distribution:
bp_cycle
6th     2364
5th     1709
None      25

co_cycle distribution:
co_cycle
6th     1928
5th     1165
None    1005

Projection period flags (bp): 363 rows
Projection period flags (co): 257 rows


## Cell 8 — Generate Table A2 per CY 2018-2025

A project appears in the year's Table A2 if it has a BP or CO event in that year. The entitlement columns (`ENT_*`) and the per-income-tier breakdown columns are not populated — CPRA does not carry that data.

`UNIT_CAT` is derived from `bp_units`: ADU if the master is flagged as an ADU permit, else 5+/2-4/SFD by unit count.

In [9]:
def unit_cat(bp_units, is_adu):
    if is_adu: return "ADU"
    u = bp_units or 0
    if u >= 5: return "5+"
    if u >= 2: return "2-4"
    return "SFD"

for year in range(2018, 2026):
    rows = []
    for _, r in proj_df.iterrows():
        has_bp = (r["reporting_year_bp"] == year)
        has_co = (r["reporting_year_co"] == year)
        if not (has_bp or has_co):
            continue
        rows.append({
            "JURIS_NAME": "BERKELEY",
            "CNTY_NAME": "Alameda",
            "YEAR": year,
            "APN": r["apn"],
            "STREET_ADDRESS": r["address"],
            "JURS_TRACKING_ID": r["master_permit"],
            "UNIT_CAT": unit_cat(r["bp_units"], r["is_adu"]),
            "TENURE": "Renter",
            "BP_ABOVE_MOD_INCOME": r["bp_units"] if has_bp else 0,
            "BP_ISSUE_DT1": r["bp_issue_date"].strftime("%Y-%m-%d") if (has_bp and r["bp_issue_date"]) else "",
            "NO_BUILDING_PERMITS": 1 if not has_bp else 0,
            "CO_ABOVE_MOD_INCOME": r["co_units"] if has_co else 0,
            "CO_ISSUE_DT1": r["co_date"].strftime("%Y-%m-%d") if (has_co and r["co_date"]) else "",
            "WORK_TYPE": r["master_work_type"],
            "WORK_DESCRIPTION": r["work_description"],
            "v2_project_id": r["v2_project_id"] or "",
            "v2_stage": r["v2_stage"] or "",
            "in_v2": r["in_v2"],
            "ADU_FLAG": "Yes" if r["is_adu"] else "No",
            # Cycle-aware columns from scripts.housing_rules (additive per Phase B)
            "bp_cycle": r["bp_cycle"] if has_bp else "",
            "co_cycle": r["co_cycle"] if has_co else "",
            "bp_in_projection_period": bool(r["bp_in_projection_period"]) if has_bp else False,
            "co_in_projection_period": bool(r["co_in_projection_period"]) if has_co else False,
        })
    out_df = pd.DataFrame(rows)
    out_path = OUT_DIR / f"table_a2_CY{year}.csv"
    out_df.to_csv(out_path, index=False)
    print(f"CY{year}: {len(out_df)} project-rows -> {out_path.name}")

CY2018: 411 project-rows -> table_a2_CY2018.csv
CY2019: 394 project-rows -> table_a2_CY2019.csv


CY2020: 333 project-rows -> table_a2_CY2020.csv


CY2021: 414 project-rows -> table_a2_CY2021.csv
CY2022: 642 project-rows -> table_a2_CY2022.csv


CY2023: 929 project-rows -> table_a2_CY2023.csv


CY2024: 1160 project-rows -> table_a2_CY2024.csv
CY2025: 1177 project-rows -> table_a2_CY2025.csv


## Cell 9 — Comparison summary

For CY 2025, load the v1-derived baseline at `data/apr/2025/table_a2_2025.csv` (the only year with a parallel v1/v2 output) and produce an APN-level diff.

For CY 2023 and 2024, only summary counts are available since no parallel CSVs exist.

In [10]:
import csv

def load_v1_baseline(path):
    if not path.exists():
        return None
    return pd.read_csv(path)

v1_2025 = load_v1_baseline(V1_APR_2025)
cpra_2025 = pd.read_csv(OUT_DIR / "table_a2_CY2025.csv")

print(f"CPRA CY2025: {len(cpra_2025)} rows")

if v1_2025 is not None:
    print(f"v1 (legacy) CY2025: {len(v1_2025)} rows")
    print()
    print("v1 columns:", list(v1_2025.columns))
    # Align on APN if v1 has one
    apn_col = next((c for c in v1_2025.columns if "apn" in c.lower()), None)
    if apn_col:
        v1_apns_2025 = set(v1_2025[apn_col].dropna().astype(str).str.strip())
        cpra_apns_2025 = set(cpra_2025["APN"].dropna().astype(str).str.strip())
        only_v1 = v1_apns_2025 - cpra_apns_2025
        only_cpra = cpra_apns_2025 - v1_apns_2025
        both = v1_apns_2025 & cpra_apns_2025
        print(f"  v1-only APNs in 2025:  {len(only_v1)}")
        print(f"  CPRA-only APNs in 2025: {len(only_cpra)}")
        print(f"  in both:               {len(both)}")
        # Sample
        print()
        print("Sample v1-only APNs (5):", list(only_v1)[:5])
        print("Sample CPRA-only APNs (5):", list(only_cpra)[:5])
else:
    print("v1 CY2025 baseline not found; skipping diff.")

print()
print("=== Summary counts for CY 2023/2024/2025 ===")
for yr in [2023, 2024, 2025]:
    df_yr = pd.read_csv(OUT_DIR / f"table_a2_CY{yr}.csv")
    print(f"CY{yr}: {len(df_yr)} CPRA-derived projects; "
          f"in_v2={df_yr['in_v2'].sum()}, "
          f"adu={int((df_yr['ADU_FLAG'] == 'Yes').sum())}, "
          f"5+_units={(df_yr['UNIT_CAT'] == '5+').sum()}")

CPRA CY2025: 1177 rows
v1 (legacy) CY2025: 27 rows

v1 columns: ['id', 'address', 'apn', 'permits', 'net_units', 'vli_units', 'status', 'app_filed_date', 'app_complete_date', 'entitled_date', 'bp_issued_date', 'co_date', 'density_bonus', 'sb35_flag', 'sb330_flag', 'ab2011_flag', 'developer', 'architect', 'construction_status', 'milestone_achieved']
  v1-only APNs in 2025:  16
  CPRA-only APNs in 2025: 1165
  in both:               8

Sample v1-only APNs (5): ['057 202901500', '057 202201701', '057 205300200', '059227901600', '055 182601802']
Sample CPRA-only APNs (5): ['054 180201400', '060 242902400', '055 184102400', '062 289101200', '056 192700300']

=== Summary counts for CY 2023/2024/2025 ===
CY2023: 929 CPRA-derived projects; in_v2=10, adu=74, 5+_units=14
CY2024: 1160 CPRA-derived projects; in_v2=12, adu=112, 5+_units=11
CY2025: 1177 CPRA-derived projects; in_v2=26, adu=115, 5+_units=17


## Cell 10 — Audit log

Writes the complete per-project derivation record to `output/D5/audit_CY2018-2025.csv`. Every CPRA-derived project, its master permit, all derivations, and join flags.

In [11]:
audit_path = OUT_DIR / "audit_CY2018-2025.csv"
proj_df.to_csv(audit_path, index=False)
print(f"wrote audit: {audit_path.name} ({len(proj_df)} rows)")
print()
print("Sample 5 audit rows:")
audit_cols = ["apn", "master_permit", "address", "master_work_type", "is_adu",
              "bp_issue_date", "bp_units", "co_date", "co_units",
              "reporting_year_bp", "reporting_year_co", "in_v2", "v2_stage"]
print(proj_df[audit_cols].head(5).to_string())

wrote audit: audit_CY2018-2025.csv (4098 rows)

Sample 5 audit rows:
             apn master_permit            address     master_work_type  is_adu bp_issue_date  bp_units    co_date  co_units  reporting_year_bp  reporting_year_co  in_v2 v2_stage
0  048H766303800   B2023-04283    111 ALVARADO Rd           Alteration   False    2023-08-21       0.0 2024-05-21       0.0             2023.0             2024.0  False     None
1  048H767003100   B2024-06060     59 ALVARADO Rd           Alteration   False    2024-12-27       0.0        NaT       NaN             2024.0                NaN  False     None
2  048H768002301   B2023-02475   25 TANGLEWOOD Rd           Alteration   False    2023-05-24       0.0        NaT       NaN             2023.0                NaN  False     None
3  048H769000300   B2025-03505      2951 DERBY St           Alteration   False    2025-10-27       0.0        NaT       NaN             2025.0                NaN  False     None
4  052 136400200   B2024-02003  1916 ALCA